# Linguagens de Programação para Engenharia de Dados
## Encontro 4 — Notebook Prático: Processamento de dados em escala

**Instituição:** Universidade de Fortaleza (UNIFOR)
**Curso:** Pós-Graduação — Especialização em Engenharia de Dados
**Professor:** Cassio Pinheiro — [LinkedIn](https://www.linkedin.com/in/cassio-pinheiro-9baa7939/)
**Data:** 09/07/2026

---

Hoje **escalamos** o nosso dataset de vendas para mais de **200 mil transações** (sintéticas e determinísticas, com `seed=42`) para *sentir* o problema da escala. Vamos comparar CSV vs. Parquet, processar em *chunks*, usar Polars *lazy* e ver o código distribuído (PySpark) de forma ilustrativa.

> Cada seção do notebook tem a marca **📖 No `.md`:** apontando para a seção correspondente do material teórico.

## Setup

Núcleo **obrigatório**: `pandas` e `numpy`. Tentamos importar `polars` e `pyarrow` (instaláveis: `pip install polars pyarrow`); se faltarem, as células correspondentes avisam com clareza. **PySpark é opcional** e tratado em `try/except` — não é necessário instalá-lo.

In [1]:
import pandas as pd
import numpy as np
import os, time, csv

# Bibliotecas opcionais do ecossistema de escala
try:
    import pyarrow            # noqa: F401  (usado pelo pandas para Parquet)
    TEM_PYARROW = True
except ImportError:
    TEM_PYARROW = False

try:
    import polars as pl
    TEM_POLARS = True
except ImportError:
    TEM_POLARS = False

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("pyarrow:", "OK" if TEM_PYARROW else "ausente (pip install pyarrow)")
print("polars :", pl.__version__ if TEM_POLARS else "ausente (pip install polars)")

# Pasta de trabalho para os arquivos gerados
PASTA = "dados_e4"
os.makedirs(PASTA, exist_ok=True)
print("\nArquivos serão gravados em:", os.path.abspath(PASTA))

pandas : 2.3.3
numpy  : 2.2.6
pyarrow: OK
polars : 1.42.0

Arquivos serão gravados em: /tmp/dados_e4


---

# 1 — Sentindo a escala

> 📖 **No `.md`:** seção *1 — O problema da escala*.

Vamos gerar **200 mil transações** sintéticas e determinísticas. O segredo do "determinístico" é `np.random.seed(42)`: toda vez que este notebook roda, os números são **exatamente os mesmos** — reprodutibilidade, um valor central de Engenharia de Dados.

In [2]:
N = 200_000  # duzentas mil transações — grande o bastante para "sentir" a escala

np.random.seed(42)  # determinismo: mesmos dados em toda execução

categorias = np.array(["eletronicos", "alimentacao", "vestuario",
                       "casa", "esporte", "livros"])
cidades = np.array(["Fortaleza", "Recife", "Natal", "Salvador",
                    "Sao Paulo", "Rio de Janeiro"])

# Datas espalhadas ao longo de ~3 meses (para depois particionar por data)
datas_base = pd.date_range("2026-04-01", "2026-06-30", freq="D")

df = pd.DataFrame({
    "id":        np.arange(1, N + 1),
    "data":      np.random.choice(datas_base, size=N).astype("datetime64[ns]"),
    "cliente_id":np.random.randint(1, 20_000, size=N),
    "cidade":    np.random.choice(cidades, size=N),
    "categoria": np.random.choice(categorias, size=N, p=[.25,.30,.15,.12,.10,.08]),
    "quantidade":np.random.randint(1, 6, size=N),
    "valor":     np.round(np.random.gamma(2.0, 60.0, size=N), 2),
})

print(f"Linhas: {len(df):,}".replace(",", "."))
print(f"Colunas: {list(df.columns)}")
df.head()

Linhas: 200.000
Colunas: ['id', 'data', 'cliente_id', 'cidade', 'categoria', 'quantidade', 'valor']


,id,data,cliente_id,cidade,categoria,quantidade,valor
0,1,2026-05-22,9120,Recife,eletronicos,4,179.31
1,2,2026-04-15,17714,Recife,eletronicos,4,103.73
2,3,2026-06-11,17113,Fortaleza,casa,1,271.54
3,4,2026-05-31,83,Recife,alimentacao,4,106.61
4,5,2026-04-21,6611,Natal,casa,2,299.52


Quanto esse DataFrame ocupa **em memória**? Esse é o número que decide se "carregar tudo" é viável.

In [3]:
mem_bytes = df.memory_usage(deep=True).sum()
print(f"Memória ocupada pelo DataFrame: {mem_bytes/1e6:,.1f} MB")
print(f"Por linha: ~{mem_bytes/len(df):.0f} bytes")
print()
print("Imagine multiplicar por 1.000x (200 milhões de linhas):",
      f"~{mem_bytes*1000/1e9:,.1f} GB — não cabe na RAM de um laptop.")
print("É exatamente aí que as técnicas de hoje deixam de ser luxo e viram necessidade.")

Memória ocupada pelo DataFrame: 34.3 MB
Por linha: ~172 bytes

Imagine multiplicar por 1.000x (200 milhões de linhas): ~34.3 GB — não cabe na RAM de um laptop.
É exatamente aí que as técnicas de hoje deixam de ser luxo e viram necessidade.


---

# 2 — CSV vs. Parquet: medindo a diferença

> 📖 **No `.md`:** seção *2 — Formatos colunares*.

Vamos gravar **o mesmo dataset** em CSV (formato por linha, texto) e em Parquet (colunar, comprimido) e **medir**: tamanho em disco e tempo de leitura. Nada de "acredite em mim" — números reais.

In [4]:
caminho_csv = os.path.join(PASTA, "vendas_grande.csv")
caminho_pq  = os.path.join(PASTA, "vendas_grande.parquet")

# Gravar CSV
t0 = time.perf_counter()
df.to_csv(caminho_csv, index=False)
t_escrita_csv = time.perf_counter() - t0

# Gravar Parquet (requer pyarrow)
if TEM_PYARROW:
    t0 = time.perf_counter()
    df.to_parquet(caminho_pq, engine="pyarrow", index=False)
    t_escrita_pq = time.perf_counter() - t0
else:
    t_escrita_pq = None
    print("pyarrow ausente — pulei a escrita do Parquet. Instale com: pip install pyarrow")

tam_csv = os.path.getsize(caminho_csv)
tam_pq  = os.path.getsize(caminho_pq) if TEM_PYARROW else None

print(f"CSV     : {tam_csv/1e6:6.2f} MB  (escrita {t_escrita_csv:.2f}s)")
if TEM_PYARROW:
    print(f"Parquet : {tam_pq/1e6:6.2f} MB  (escrita {t_escrita_pq:.2f}s)")
    print(f"\nParquet é {tam_csv/tam_pq:.1f}x menor em disco que o CSV.")

CSV     :  10.17 MB  (escrita 0.20s)
Parquet :   2.48 MB  (escrita 0.04s)

Parquet é 4.1x menor em disco que o CSV.


Agora o que mais importa em pipelines de escala: **tempo de leitura**. E o ganho extra do colunar — ler **só as colunas necessárias** (*column pushdown*).

In [5]:
# Leitura COMPLETA do CSV
t0 = time.perf_counter()
_ = pd.read_csv(caminho_csv)
t_leitura_csv = time.perf_counter() - t0
print(f"Ler CSV completo            : {t_leitura_csv:.3f}s")

if TEM_PYARROW:
    # Leitura COMPLETA do Parquet
    t0 = time.perf_counter()
    _ = pd.read_parquet(caminho_pq, engine="pyarrow")
    t_leitura_pq = time.perf_counter() - t0
    print(f"Ler Parquet completo        : {t_leitura_pq:.3f}s   ({t_leitura_csv/t_leitura_pq:.1f}x mais rápido)")

    # Leitura de APENAS 2 colunas do Parquet — column pushdown
    t0 = time.perf_counter()
    _ = pd.read_parquet(caminho_pq, engine="pyarrow", columns=["categoria", "valor"])
    t_leitura_pq2col = time.perf_counter() - t0
    print(f"Ler Parquet (só 2 colunas)  : {t_leitura_pq2col:.3f}s   <- lê apenas o que precisa do disco")

Ler CSV completo            : 0.056s
Ler Parquet completo        : 0.012s   (4.5x mais rápido)
Ler Parquet (só 2 colunas)  : 0.006s   <- lê apenas o que precisa do disco


> **O que observar:** o Parquet é menor *e* mais rápido de ler. E quando você pede **só duas colunas**, ele lê apenas esses blocos do disco — o CSV nunca conseguiria isso, pois precisaria varrer cada linha inteira. Esse é o *column pushdown* em ação.

---

# 3 — Processamento em lote (chunks)

> 📖 **No `.md`:** seção *3 — Particionamento e processamento em lote*.

E se o arquivo **não coubesse** na memória? A resposta clássica: ler em **pedaços** (`chunksize`), processar cada pedaço, **acumular o resultado parcial** e descartar o pedaço. A memória usada fica limitada a um *chunk* — não ao arquivo inteiro.

Vamos somar o faturamento por categoria **sem nunca carregar o CSV inteiro**.

In [6]:
faturamento = {}          # acumulador
linhas_vistas = 0
n_chunks = 0

TAMANHO_CHUNK = 50_000    # processa 50 mil linhas por vez

for bloco in pd.read_csv(caminho_csv, chunksize=TAMANHO_CHUNK):
    n_chunks += 1
    linhas_vistas += len(bloco)
    # resultado PARCIAL deste bloco
    parcial = bloco.groupby("categoria")["valor"].sum()
    # COMBINA com o acumulado (soma é incremental!)
    for categoria, valor in parcial.items():
        faturamento[categoria] = faturamento.get(categoria, 0.0) + valor

print(f"Processados {n_chunks} chunks, {linhas_vistas:,} linhas".replace(",", "."),
      f"(máx. {TAMANHO_CHUNK:,} linhas na memória por vez)".replace(",", "."))
print("\nFaturamento por categoria (via chunks):")
for cat, val in sorted(faturamento.items(), key=lambda x: -x[1]):
    print(f"  {cat:12s}: R$ {val:14,.2f}".replace(",", "."))

Processados 4 chunks. 200.000 linhas (máx. 50.000 linhas na memória por vez)

Faturamento por categoria (via chunks):
  alimentacao : R$   7.228.649.50
  eletronicos : R$   5.997.182.78
  vestuario   : R$   3.565.544.55
  casa        : R$   2.879.478.79
  esporte     : R$   2.384.849.65
  livros      : R$   1.929.240.33


Vamos **conferir** que o resultado em chunks bate com a agregação direta (que carrega tudo). O padrão "parcial + combina" tem que dar o mesmo número.

In [7]:
conferencia = df.groupby("categoria")["valor"].sum().round(2)
em_chunks   = pd.Series(faturamento).round(2)

ok = (conferencia.sort_index().values.round(2)
      == em_chunks.sort_index().values.round(2)).all()
print("Resultado em chunks == resultado direto?", ok)
print("\nO chunking processou o MESMO dado usando uma fração da memória.")

Resultado em chunks == resultado direto? True

O chunking processou o MESMO dado usando uma fração da memória.


**Bônus — particionar por data:** gravar uma pasta por dia. Assim, uma consulta que filtra um dia lê só aquela pasta (*partition pruning*).

In [8]:
if TEM_PYARROW:
    pasta_particao = os.path.join(PASTA, "vendas_particionado")
    df["data_str"] = df["data"].dt.strftime("%Y-%m-%d")
    df.to_parquet(pasta_particao, engine="pyarrow",
                  partition_cols=["categoria"], index=False)
    # Quantas "pastas" (partições) foram criadas?
    parts = [p for p in os.listdir(pasta_particao) if p.startswith("categoria=")]
    print(f"Partições criadas (uma por categoria): {len(parts)}")
    for p in sorted(parts):
        print("   ", p)
    df.drop(columns=["data_str"], inplace=True)
    print("\nFiltrar por uma categoria agora lê só a pasta daquela categoria (partition pruning).")
else:
    print("pyarrow ausente — particionamento em Parquet pulado.")

Partições criadas (uma por categoria): 6
    categoria=alimentacao
    categoria=casa
    categoria=eletronicos
    categoria=esporte
    categoria=livros
    categoria=vestuario

Filtrar por uma categoria agora lê só a pasta daquela categoria (partition pruning).


---

# 4 — Polars lazy: deixe o motor otimizar

> 📖 **No `.md`:** seção *4 — Avaliação tardia (lazy)*.

O pandas é **eager**: executa cada passo na hora. O **Polars lazy** apenas **registra a intenção** com `scan_parquet`, monta um **plano**, e só executa no `collect()` — otimizando o plano inteiro (lê só as colunas e linhas necessárias).

Vamos rodar a **mesma agregação** (faturamento por categoria) em pandas e em Polars lazy e comparar o tempo.

In [9]:
# Referência: pandas eager (lê tudo, depois agrega)
t0 = time.perf_counter()
res_pandas = (pd.read_parquet(caminho_pq, engine="pyarrow")
              .groupby("categoria")["valor"].sum()
              .sort_values(ascending=False)) if TEM_PYARROW else None
t_pandas = time.perf_counter() - t0 if TEM_PYARROW else None
if TEM_PYARROW:
    print(f"pandas (eager): {t_pandas:.3f}s")
    print(res_pandas.round(2), "\n")

pandas (eager): 0.016s
categoria
alimentacao    7228649.50
eletronicos    5997182.78
vestuario      3565544.55
casa           2879478.79
esporte        2384849.65
livros         1929240.33
Name: valor, dtype: float64 



In [10]:
if TEM_POLARS and TEM_PYARROW:
    # Polars LAZY: scan_parquet não lê nada; o plano só roda no collect()
    plano = (
        pl.scan_parquet(caminho_pq)          # lazy: registra a fonte, não lê
          .group_by("categoria")
          .agg(pl.col("valor").sum().alias("faturamento"))
          .sort("faturamento", descending=True)
    )

    print("=== Plano de consulta otimizado (Polars decidiu o que ler) ===")
    print(plano.explain())                   # mostra a otimização ANTES de executar
    print()

    t0 = time.perf_counter()
    res_polars = plano.collect()             # AGORA executa o plano otimizado
    t_polars = time.perf_counter() - t0
    print(f"Polars (lazy): {t_polars:.3f}s")
    print(res_polars)

    if t_pandas:
        print(f"\nPolars foi ~{t_pandas/max(t_polars,1e-6):.1f}x o tempo do pandas nesta máquina.")
        print("(O ganho cresce muito com o volume e com filtros que o lazy empurra para a leitura.)")
elif not TEM_POLARS:
    print("Polars ausente — instale com: pip install polars")
    print("Equivalente lazy seria:")
    print("  pl.scan_parquet(arquivo).group_by('categoria')"
          ".agg(pl.col('valor').sum()).collect()")
else:
    print("pyarrow ausente — Parquet indisponível para o exemplo Polars.")

=== Plano de consulta otimizado (Polars decidiu o que ler) ===
SORT BY [descending: [true]] [col("faturamento")]
  AGGREGATE[maintain_order: false]
    [col("valor").sum().alias("faturamento")] BY [col("categoria")]
    FROM
    Parquet SCAN [dados_e4/vendas_grande.parquet]
    PROJECT 2/7 COLUMNS
    ESTIMATED ROWS: 200000

Polars (lazy): 0.004s
shape: (6, 2)
┌─────────────┬─────────────┐
│ categoria   ┆ faturamento │
│ ---         ┆ ---         │
│ str         ┆ f64         │
╞═════════════╪═════════════╡
│ alimentacao ┆ 7.2286e6    │
│ eletronicos ┆ 5.9972e6    │
│ vestuario   ┆ 3.5655e6    │
│ casa        ┆ 2.8795e6    │
│ esporte     ┆ 2.3848e6    │
│ livros      ┆ 1.9292e6    │
└─────────────┴─────────────┘

Polars foi ~4.0x o tempo do pandas nesta máquina.
(O ganho cresce muito com o volume e com filtros que o lazy empurra para a leitura.)


Onde o *lazy* realmente brilha: **filtrar + agregar**. O Polars empurra o filtro para a leitura (*predicate pushdown*) e só carrega as colunas necessárias (*projection pushdown*) — lê uma **fração** do dado.

In [11]:
if TEM_POLARS and TEM_PYARROW:
    recorte = (
        pl.scan_parquet(caminho_pq)
          .filter(pl.col("categoria") == "eletronicos")   # empurrado p/ leitura
          .select(["categoria", "valor"])                 # só 2 colunas
          .group_by("categoria")
          .agg([pl.col("valor").sum().alias("faturamento"),
                pl.len().alias("n_transacoes")])
          .collect()
    )
    print(recorte)
    print("\nO motor leu só as colunas 'categoria' e 'valor' e descartou as outras categorias na leitura.")
else:
    print("Exemplo de filtro+agregação lazy requer polars + pyarrow.")

shape: (1, 3)
┌─────────────┬─────────────┬──────────────┐
│ categoria   ┆ faturamento ┆ n_transacoes │
│ ---         ┆ ---         ┆ ---          │
│ str         ┆ f64         ┆ u32          │
╞═════════════╪═════════════╪══════════════╡
│ eletronicos ┆ 5.9972e6    ┆ 49873        │
└─────────────┴─────────────┴──────────────┘

O motor leu só as colunas 'categoria' e 'valor' e descartou as outras categorias na leitura.


---

# 5 — PySpark (ilustrativo): processamento distribuído

> 📖 **No `.md`:** seção *5 — Processamento distribuído*.

Quando nem a maior máquina basta, distribui-se o trabalho num **cluster** com **Apache Spark**. Spark é **pesado** — **não vamos instalá-lo** aqui. A célula abaixo tenta importar PySpark; se não houver (o caso esperado), ela **cai no fallback**: imprime o código equivalente e explica **transformações vs. ações**, **sem erro**.

In [12]:
def explicar_pyspark():
    """Fallback didático: mostra o código Spark e explica a execução lazy/DAG."""
    codigo = '''
    from pyspark.sql import SparkSession, functions as F

    spark = SparkSession.builder.appName("vendas").getOrCreate()

    # LER -> transformação (LAZY): nada é computado ainda
    df = spark.read.parquet("vendas_grande.parquet")

    resultado = (
        df.filter(F.col("categoria") == "eletronicos")   # transformação (lazy)
          .groupBy("categoria")                           # transformação (lazy)
          .agg(F.sum("valor").alias("faturamento"))       # transformação (lazy)
    )

    resultado.show()   # AÇÃO -> só AQUI o Spark monta o DAG e dispara o cluster
    '''
    print(codigo)
    print("Como o Spark executa isto:")
    print("  • TRANSFORMAÇÕES (filter, groupBy, agg, select, join) NÃO executam nada —")
    print("    apenas adicionam nós a um plano (DAG). São LAZY, como o Polars.")
    print("  • AÇÕES (show, count, collect, write) pedem um resultado concreto —")
    print("    só AQUI o Spark otimiza o DAG (Catalyst) e distribui o job pelas máquinas.")
    print("  • groupBy/join exigem reunir linhas da mesma chave em uma máquina: isso gera")
    print("    SHUFFLE (dados viajando pela rede) — a operação mais cara do mundo distribuído.")

try:
    from pyspark.sql import SparkSession, functions as F
    print("PySpark detectado — montando uma sessão local de demonstração...")
    spark = SparkSession.builder.master("local[*]").appName("vendas_e4").getOrCreate()
    sdf = spark.read.parquet(caminho_pq)                 # transformação (lazy)
    out = (sdf.filter(F.col("categoria") == "eletronicos")
              .groupBy("categoria")
              .agg(F.sum("valor").alias("faturamento")))  # tudo lazy até aqui
    out.show()                                            # AÇÃO: dispara a execução
    spark.stop()
except Exception as e:
    print(f"PySpark não disponível ({type(e).__name__}). Usando o fallback ilustrativo.\n")
    explicar_pyspark()

PySpark não disponível (ModuleNotFoundError). Usando o fallback ilustrativo.


    from pyspark.sql import SparkSession, functions as F

    spark = SparkSession.builder.appName("vendas").getOrCreate()

    # LER -> transformação (LAZY): nada é computado ainda
    df = spark.read.parquet("vendas_grande.parquet")

    resultado = (
        df.filter(F.col("categoria") == "eletronicos")   # transformação (lazy)
          .groupBy("categoria")                           # transformação (lazy)
          .agg(F.sum("valor").alias("faturamento"))       # transformação (lazy)
    )

    resultado.show()   # AÇÃO -> só AQUI o Spark monta o DAG e dispara o cluster
    
Como o Spark executa isto:
  • TRANSFORMAÇÕES (filter, groupBy, agg, select, join) NÃO executam nada —
    apenas adicionam nós a um plano (DAG). São LAZY, como o Polars.
  • AÇÕES (show, count, collect, write) pedem um resultado concreto —
    só AQUI o Spark otimiza o DAG (Catalyst) e distribui o job pelas máquinas.
  • groupBy/

> **Repare na simetria:** Polars (`scan`/`collect`) e Spark (transformações/ações) compartilham a **mesma ideia** — construir um plano e otimizá-lo antes de executar. A diferença é a escala: Polars otimiza para **uma máquina potente**; Spark, para **um cluster** — ao custo do *shuffle*.

---

# 6 — Qual ferramenta usar

> 📖 **No `.md`:** seção *6 — Qual ferramenta usar*.

Não existe "a melhor" — existe a **certa para o tamanho do problema**. Vamos consolidar o mapa de decisão e olhar nossos próprios números.

In [13]:
mapa = pd.DataFrame([
    {"tamanho": "Cabe folgado na RAM (até ~poucos GB)",
     "ferramenta": "pandas",
     "porque": "ecossistema, familiaridade, cola com tudo"},
    {"tamanho": "Grande, mas cabe num servidor potente (dezenas-centenas de GB)",
     "ferramenta": "Polars / DuckDB",
     "porque": "single-node, lazy, multi-core, sem overhead de cluster"},
    {"tamanho": "Grande demais para qualquer máquina (TB-PB)",
     "ferramenta": "Spark (distribuído)",
     "porque": "escala horizontal real, tolerância a falhas"},
])
print("MAPA DE DECISÃO — qual ferramenta para qual escala:\n")
for _, r in mapa.iterrows():
    print(f"• {r['tamanho']}")
    print(f"    -> {r['ferramenta']}  ({r['porque']})\n")

MAPA DE DECISÃO — qual ferramenta para qual escala:

• Cabe folgado na RAM (até ~poucos GB)
    -> pandas  (ecossistema, familiaridade, cola com tudo)

• Grande, mas cabe num servidor potente (dezenas-centenas de GB)
    -> Polars / DuckDB  (single-node, lazy, multi-core, sem overhead de cluster)

• Grande demais para qualquer máquina (TB-PB)
    -> Spark (distribuído)  (escala horizontal real, tolerância a falhas)



In [14]:
print("=== Nossos números deste notebook (200 mil linhas) ===")
print(f"CSV em disco     : {tam_csv/1e6:6.2f} MB")
if TEM_PYARROW:
    print(f"Parquet em disco : {tam_pq/1e6:6.2f} MB   ({tam_csv/tam_pq:.1f}x menor)")
    print(f"Ler CSV          : {t_leitura_csv:.3f}s")
    print(f"Ler Parquet      : {t_leitura_pq:.3f}s   ({t_leitura_csv/t_leitura_pq:.1f}x mais rápido)")
print()
print("Conclusão de engenharia:")
print("  Para 200 mil — ou até dezenas de milhões — de linhas, NÃO precisamos de cluster.")
print("  Parquet + Polars/DuckDB numa única máquina resolvem com folga.")
print("  Lakehouse (Delta Lake / Iceberg) é o que une 'lago barato' + 'garantias de banco'")
print("  sobre esses mesmos arquivos Parquet abertos. Esse é o padrão dominante de 2026.")

=== Nossos números deste notebook (200 mil linhas) ===
CSV em disco     :  10.17 MB
Parquet em disco :   2.48 MB   (4.1x menor)
Ler CSV          : 0.056s
Ler Parquet      : 0.012s   (4.5x mais rápido)

Conclusão de engenharia:
  Para 200 mil — ou até dezenas de milhões — de linhas, NÃO precisamos de cluster.
  Parquet + Polars/DuckDB numa única máquina resolvem com folga.
  Lakehouse (Delta Lake / Iceberg) é o que une 'lago barato' + 'garantias de banco'
  sobre esses mesmos arquivos Parquet abertos. Esse é o padrão dominante de 2026.


---

## 🧪 Exercícios do Encontro 4

Use o `df` (e os arquivos `vendas_grande.csv` / `.parquet`) já criados acima.

1. **Tamanho e escala.** Recrie o dataset com `N = 1_000_000` (um milhão) e refaça a comparação CSV vs. Parquet (tamanho em disco e tempo de leitura). O ganho do Parquet aumenta, diminui ou se mantém proporcional? Por quê?

2. **Chunking incremental.** Adapte o laço da Seção 3 para calcular, **em chunks**, o **ticket médio** (faturamento ÷ número de transações) por **cidade**. Dica: você precisa acumular **dois** parciais — soma de `valor` e contagem — e dividir só no final. Por que a média **não** pode ser somada diretamente entre chunks?

3. **Polars lazy com filtro.** Escreva uma consulta Polars *lazy* que retorne, **apenas para a cidade "Fortaleza"**, o faturamento total por categoria, ordenado do maior para o menor. Use `scan_parquet`, `filter`, `group_by`/`agg` e `collect()`. Rode `.explain()` antes do `collect()` e identifique onde o filtro foi empurrado.

4. **Transformações vs. ações.** No bloco de código PySpark da Seção 5, classifique **cada linha** como *transformação* (lazy) ou *ação* (dispara execução). Em seguida, explique em uma frase por que a operação `groupBy` provoca um **shuffle** num cluster.

> Guarde suas respostas: alguns destes conceitos voltam no Encontro 5, quando ligarmos **escala** a **qualidade de dados**.

## Encerramento

Hoje saímos do "dado que cabe na memória" para o **dado em escala**:

- **sentimos** o problema gerando 200 mil transações e medindo a memória;
- comparamos **CSV vs. Parquet** com números reais (tamanho e tempo de leitura);
- processamos **em chunks**, sem carregar tudo, e particionamos por chave;
- usamos **Polars lazy** (`scan_parquet`/`collect()`) e vimos o **plano otimizado**;
- entendemos o **distribuído** (PySpark: transformações vs. ações, DAG, shuffle) de forma ilustrativa;
- consolidamos o **mapa de decisão**: pandas -> Polars/DuckDB -> Spark, e o **Lakehouse** abertos.

**Princípios que ficam:** colunar, lazy, particionar — e, acima de tudo, **não processe o que você não precisa ler**.

No **Encontro 5**, voltamos ao dado de perto: **qualidade de dados** — porque dado grande e rápido só vale se for **confiável**.

---
*UNIFOR Pós-Graduação · Prof. Cassio Pinheiro · Acompanha o material `e4_material.md` e os slides `e4_slides.pptx`.*